# MQTT Monitor Notebook

Use this notebook to monitor live MQTT communication for the workshop agents.

Default behavior:
- Connect using `simulated_city.config.load_config()`
- Subscribe to `<base_topic>/sim/#`
- Show latest messages with topic + payload

In [1]:
# Cell purpose: import dependencies and connect to MQTT using project config.
from collections import deque
from datetime import datetime
import time

from IPython.display import Markdown, clear_output, display

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector

config = load_config()
print(f"Loaded config. Primary MQTT profile: {config.mqtt.host}:{config.mqtt.port}")

connector = MqttConnector(config.mqtt, client_id_suffix="mqtt-monitor")
connector.connect()
connected = connector.wait_for_connection(timeout=5)
print(f"MQTT connected: {connected}")

Loaded config. Primary MQTT profile: 127.0.0.1:1883
MQTT connected: True


In [2]:
# Cell purpose: configure subscription target and callback state.
base_topic = config.mqtt.base_topic.rstrip("/")
topic_filter = f"{base_topic}/sim/#"
max_rows = 25
messages = deque(maxlen=max_rows)
message_count = 0

def _safe_decode(raw_payload):
    try:
        return raw_payload.decode("utf-8")
    except Exception:
        return repr(raw_payload)

def on_message(client, userdata, msg):
    global message_count
    message_count += 1
    messages.append({
        "index": message_count,
        "time": datetime.now().isoformat(timespec="seconds"),
        "topic": msg.topic,
        "payload": _safe_decode(msg.payload),
    })

connector.client.on_message = on_message
connector.client.subscribe(topic_filter, qos=0)
print(f"Monitoring topic filter: {topic_filter}")

Monitoring topic filter: simulated-city/sim/#


In [ ]:
# Cell purpose: keep notebook alive and render latest MQTT messages.
print("MQTT monitor loop running. Interrupt the cell to stop.")
try:
    while True:
        clear_output(wait=True)
        lines = [
            "### MQTT Monitor",
            f"- Filter: `{topic_filter}`",
            f"- Total messages: {message_count}",
            f"- Showing latest: {len(messages)}",
            "",
            "| # | Time | Topic | Payload |",
            "|---:|---|---|---|",
        ]

        for item in reversed(messages):
            payload = str(item["payload"]).replace("|", "\\|")
            topic = str(item["topic"]).replace("|", "\\|")
            lines.append(f"| {item['index']} | {item['time']} | {topic} | {payload} |")

        display(Markdown("\n".join(lines)))
        time.sleep(0.5)
except KeyboardInterrupt:
    print("Stopping MQTT monitor loop.")
finally:
    connector.disconnect()
    print(f"MQTT disconnected. Captured {message_count} messages.")

### MQTT Monitor
- Filter: `simulated-city/sim/#`
- Total messages: 4200
- Showing latest: 25

| # | Time | Topic | Payload |
|---:|---|---|---|
| 4200 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 300, "population": {"sheep": 62, "wolves": 8}, "energy": {"sheep_average": 8.935}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.7, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.699151+00:00"} |
| 4199 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 299, "population": {"sheep": 62, "wolves": 8}, "energy": {"sheep_average": 8.903}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.7, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.698410+00:00"} |
| 4198 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 298, "population": {"sheep": 63, "wolves": 8}, "energy": {"sheep_average": 8.873}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.71, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.695213+00:00"} |
| 4197 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 297, "population": {"sheep": 63, "wolves": 8}, "energy": {"sheep_average": 8.667}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.71, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.694473+00:00"} |
| 4196 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 296, "population": {"sheep": 60, "wolves": 8}, "energy": {"sheep_average": 9.1}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.68, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.690857+00:00"} |
| 4195 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 295, "population": {"sheep": 63, "wolves": 8}, "energy": {"sheep_average": 8.778}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.71, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.690388+00:00"} |
| 4194 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 294, "population": {"sheep": 67, "wolves": 8}, "energy": {"sheep_average": 8.537}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.75, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.689125+00:00"} |
| 4193 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 293, "population": {"sheep": 67, "wolves": 8}, "energy": {"sheep_average": 8.582}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.75, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.688608+00:00"} |
| 4192 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 292, "population": {"sheep": 67, "wolves": 8}, "energy": {"sheep_average": 8.507}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.75, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.688140+00:00"} |
| 4191 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 291, "population": {"sheep": 67, "wolves": 8}, "energy": {"sheep_average": 8.433}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.75, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.687660+00:00"} |
| 4190 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 290, "population": {"sheep": 66, "wolves": 8}, "energy": {"sheep_average": 8.53}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.74, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.682433+00:00"} |
| 4189 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 289, "population": {"sheep": 66, "wolves": 8}, "energy": {"sheep_average": 8.742}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.74, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.681366+00:00"} |
| 4188 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 288, "population": {"sheep": 64, "wolves": 8}, "energy": {"sheep_average": 8.953}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.72, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.663616+00:00"} |
| 4187 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 287, "population": {"sheep": 63, "wolves": 8}, "energy": {"sheep_average": 9.206}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.71, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.662919+00:00"} |
| 4186 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 286, "population": {"sheep": 63, "wolves": 8}, "energy": {"sheep_average": 8.683}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.71, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.655697+00:00"} |
| 4185 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 285, "population": {"sheep": 64, "wolves": 8}, "energy": {"sheep_average": 8.734}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.72, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.655133+00:00"} |
| 4184 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 284, "population": {"sheep": 65, "wolves": 8}, "energy": {"sheep_average": 8.738}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.73, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.654663+00:00"} |
| 4183 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 283, "population": {"sheep": 68, "wolves": 8}, "energy": {"sheep_average": 8.235}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.76, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.654082+00:00"} |
| 4182 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 282, "population": {"sheep": 68, "wolves": 8}, "energy": {"sheep_average": 8.294}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.76, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.645899+00:00"} |
| 4181 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 281, "population": {"sheep": 66, "wolves": 8}, "energy": {"sheep_average": 8.636}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.74, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.645448+00:00"} |
| 4180 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 280, "population": {"sheep": 69, "wolves": 8}, "energy": {"sheep_average": 8.623}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.77, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.644955+00:00"} |
| 4179 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 279, "population": {"sheep": 69, "wolves": 8}, "energy": {"sheep_average": 8.348}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.77, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.644475+00:00"} |
| 4178 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 278, "population": {"sheep": 67, "wolves": 8}, "energy": {"sheep_average": 8.821}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.75, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.640886+00:00"} |
| 4177 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 277, "population": {"sheep": 65, "wolves": 8}, "energy": {"sheep_average": 9.169}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.73, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.640449+00:00"} |
| 4176 | 2026-03-05T10:47:51 | simulated-city/sim/observer/metrics | {"tick": 276, "population": {"sheep": 66, "wolves": 8}, "energy": {"sheep_average": 8.939}, "grass": {"grown_cells": 34, "coverage_pct": 34.0}, "events": {"births": 0, "deaths": 0, "predation": 0, "sheep_ate_grass": 0, "wolf_births": 0, "wolf_deaths": 0}, "occupancy": {"estimated_ratio": 0.74, "grid_width": 10, "grid_height": 10, "method": "(sheep+wolves)/cells"}, "controller": {"latest_reason": "baseline"}, "timestamp_utc": "2026-03-05T09:47:51.640029+00:00"} |